# CNN Regressor Network

In this notebook, we will be constructing our CNN-based regression network.

In [ ]:
import numpy as np

from pathlib import Path
from tqdm import tqdm

# Torch imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Diet4Cola imports
from diet4cola.utils import load_array, plot_2d_array, plot_curve

## Preprocessing

We first make a function to gather all the filenames on disk and organize them.

In [ ]:
data_dir = Path('../data/synthetic')

In [ ]:
def gather_examples(dir: Path) -> dict:
    if dir.exists() and dir.is_dir():
        files = [f for f in dir.iterdir() if f.is_file()]
        file_dict = {}
        for file in files:
            identifier = file.name.split('_')[1]

            if identifier in file_dict.keys():
                file_dict[identifier].append(file.name)
            else:
                file_dict[identifier] = [file.name]
    
        # Control, remove any example that does not have 4 entries
        keys_to_pop = []
        for key in file_dict.keys():
            if len(file_dict[key]) != 4:
                keys_to_pop.append(key)

        for key in keys_to_pop:
            file_dict.pop(key, None)

        return file_dict
    return None

example_dictionary = gather_examples(data_dir)
print(f'Found {len(example_dictionary)} examples.')

Next, we create a class that can store our dataset.

In [ ]:
# Define data modality to train on
MODE_ACTOMYOSIN     = 0
MODE_ACTIN          = 1
MODE_MYOSIN         = 2

def mode_to_str(mode: int) -> str:
    match mode:
        case 0:
            return 'actomyosin'
        case 1:
            return 'actin'
        case 2:
            return 'myosin'
        case _:
            raise ValueError(f'{mode} is not a valid mode')

# Format string to obtain example
KEY_INPUT_STRING = lambda x, mode: f'cortex_{x}_{mode_to_str(mode)}.npy'
KEY_OUTPUT_STRING = lambda x: f'cortex_{x}_velocity_field.npy'

In [ ]:
class VelocityFieldDataset(Dataset):
    def __init__(self, data_dir: Path, example_dictionary: dict, mode: int, 
                 split: str = 'train', train_frac: float = 0.8, seed: int = 42):
        self.data_dir = data_dir
        self.example_dictionary = example_dictionary
        self.mode = mode
        self.split = split

        # Make sure split is valid
        assert self.split in ['train', 'val'], 'Split must be \'train\' or \'val\'.'

        # Split keys into train/validation -> TODO: Discuss with Casper if this is good approach for dataset splitting
        all_keys = list(example_dictionary.keys())
        np.random.seed(seed)
        np.random.shuffle(all_keys)
        split_idx = int(len(all_keys) * train_frac)
        if self.split == 'train':
            self.keys = all_keys[:split_idx]
        else:
            self.keys = all_keys[split_idx:]

        # Create examples
        self._create_examples()

    def _extract_example(self, key: str) -> tuple[np.ndarray, np.ndarray]:
        input_example_file = self.data_dir / KEY_INPUT_STRING(key, self.mode)
        output_example_file = self.data_dir / KEY_OUTPUT_STRING(key)

        input_example = load_array(str(input_example_file))
        output_example = load_array(str(output_example_file))

        return (input_example, output_example)

    def _create_examples(self):
        examples = []
        for key in tqdm(self.keys, desc=f'Loading {self.split} examples'):
            input_example, output_example = self._extract_example(key)

            # Check shape
            if input_example.shape != output_example.shape:
                raise ValueError(f'Input/output shape of examples must match!')
            
            frames = input_example.shape[0]
            for i in range(frames - 1):
                # Extract cortex/field
                cortex_before   = input_example[i, :, :]
                cortex_after    = input_example[i + 1, :, :]
                velocity_field  = output_example[i, :, :]

                # Stack to obtain one input
                cortex_input    = np.stack([cortex_before, cortex_after], axis=0)

                if np.isnan(cortex_before).any() or np.isnan(cortex_after).any() or np.isnan(velocity_field).any():
                    print(f"NaN detected at frame {i}, skipping...")
                    continue

                # Preprocess to tensor
                cortex_input    = torch.tensor(cortex_input, dtype=torch.float32)
                velocity_output  = torch.tensor(velocity_field, dtype=torch.float32)

                if torch.isnan(cortex_input).any().item() or torch.isnan(velocity_output).any().item():
                    continue

                examples.append((cortex_input, velocity_output))

        self.examples = examples

    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        if idx >= self.__len__():
            raise IndexError(f'Index {idx} is out of bounds, only {self.__len__()} examples in dataset.')
        return self.examples[idx]

        

In [ ]:
train_frac = 0.9

train_dataset = VelocityFieldDataset(data_dir, example_dictionary, MODE_ACTOMYOSIN, 'train', train_frac=train_frac)
val_dataset = VelocityFieldDataset(data_dir, example_dictionary, MODE_ACTOMYOSIN, 'val', train_frac=train_frac)

In [ ]:
print(f'Training examples: {train_dataset.__len__()}')
print(f'Validation examples: {val_dataset.__len__()}')

## Model

Now we can define our model implementation. In this notebook, we will experiment with different models. Only the best one is kept.

In [ ]:
class ResNetBlock(nn.Module):
    """Standard ResNet basic block (two 3x3 convs + shortcut)."""
    expansion = 1

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.use_proj = stride != 1 or in_ch != out_ch

        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)

        if self.use_proj:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out, inplace=True)
        return out
    

class UpsamplingBlock(nn.Module):
    """Upsample + concatenate skip connection and apply residual block"""
    
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2) # Double spatial size
        self.conv = ResNetBlock(out_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)                  # Upsammple
        x = torch.cat([x, skip], dim=1) # Concatenate skip features
        x = self.conv(x)                # Residual refinement
        return x
    

class ResNetFlow(nn.Module):
    def __init__(self, base_ch=32, out_channels=2, multi_scale=True):
        super().__init__()
        self.multi_scale = multi_scale

        # Encoder
        self.enc1       = ResNetBlock(2, base_ch, stride=1)                 # 512x512
        self.enc2       = ResNetBlock(base_ch, base_ch * 2, stride=2)       # 256x256
        self.enc3       = ResNetBlock(base_ch * 2, base_ch * 4, stride=2)   # 128x128
        self.enc4       = ResNetBlock(base_ch * 4, base_ch * 8, stride=2)   # 64x64
        self.enc5       = ResNetBlock(base_ch * 8, base_ch * 16, stride=2)  # 32x32

        # Bottleneck
        self.bottleneck = ResNetBlock(base_ch * 16, base_ch * 16, stride=1) # 32x32

        # Decoder
        self.dec5       = UpsamplingBlock(base_ch * 16, base_ch * 8, base_ch * 8)   # 32x32 -> 64x64
        self.dec4       = UpsamplingBlock(base_ch * 8, base_ch * 4, base_ch * 4)    # 64x64 -> 128x128
        self.dec3       = UpsamplingBlock(base_ch * 4, base_ch * 2, base_ch * 2)    # 128x128 -> 256x256
        self.dec2       = UpsamplingBlock(base_ch * 2, base_ch, base_ch)            # 256x256 -> 512x512
        self.dec1       = ResNetBlock(base_ch, base_ch)                             # Full-res refinement

        # Multi-scale prediction heads
        self.pred1      = nn.Conv2d(base_ch, out_channels, kernel_size=3, padding=1)
        if self.multi_scale:
            self.pred2  = nn.Conv2d(base_ch * 2, out_channels, kernel_size=3, padding=1)
            self.pred3  = nn.Conv2d(base_ch * 4, out_channels, kernel_size=3, padding=1)
            self.pred4  = nn.Conv2d(base_ch * 8, out_channels, kernel_size=3, padding=1)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.ConvTranspose2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            
    def forward(self, x):
        # Encoder
        e1      = self.enc1(x)  # (B, base_ch, 512, 512)
        e2      = self.enc2(e1) # (B, base_ch*2, 256, 256)
        e3      = self.enc3(e2) # (B, base_ch*4, 128, 128)
        e4      = self.enc4(e3) # (B, base_ch*8, 64, 64)
        e5      = self.enc5(e4) # (B, base_ch*16, 32, 32)

        # Bottleneck
        b       = self.bottleneck(e5)  # (B, base_ch*16, 32, 32)

        # Decoder
        d5      = self.dec5(b, e4)  # (B, base_ch*8, 64, 64)
        d4      = self.dec4(d5, e3) # (B, base_ch*4, 128, 128)
        d3      = self.dec3(d4, e2) # (B, base_ch*2, 256, 256)
        d2      = self.dec2(d3, e1) # (B, base_ch, 512, 512)
        d1      = self.dec1(d2)     # (B, base_ch, 512, 512)

        # Predictions
        p1      = self.pred1(d1)  # full resolution (512x512)
        if self.multi_scale:
            p2  = self.pred2(d3)  # 256x256
            p3  = self.pred3(d4)  # 128x128
            p4  = self.pred4(d5)  # 64x64

            # Upsample all to match p1
            p2  = F.interpolate(p2, size=p1.shape[-2:], mode='bilinear', align_corners=False)
            p3  = F.interpolate(p3, size=p1.shape[-2:], mode='bilinear', align_corners=False)
            p4  = F.interpolate(p4, size=p1.shape[-2:], mode='bilinear', align_corners=False)
            return [p1, p2, p3, p4]
        else:
            return p1

## Training

In [ ]:
def collate_example(batch):
    # Stack inputs and outputs
    inputs  = torch.stack([item[0] for item in batch], dim=0)
    outputs = torch.stack([item[1] for item in batch], dim=0)

    return inputs, outputs

In [ ]:
BATCH_SIZE=1

train_loader    = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_example)
val_loader      = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_example)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

**BEFORE TRAINING, MAKE SURE THE CELL ABOVE SAYS 'CUDA' (or wait for an eternity :))**

In [ ]:
# Model
model = ResNetFlow(base_ch=32, out_channels=1, multi_scale=False)
model = model.to(device)

In [ ]:
# Loss function (L1 or L2 for flow estimation)
criterion = nn.MSELoss() # Mean Squared Error (for now)

In [ ]:
# Optimizer
lr = 1e-6
optimizer = optim.Adam(model.parameters(), lr=lr)

In [ ]:
# Training loop
num_epochs = 10
train_loss_on_epoch = []
val_loss_on_epoch = []

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0

    loop = tqdm(train_loader, total=len(train_loader), desc=f'Epoch {epoch+1}/{num_epochs}')
    for inputs, targets in loop:
        inputs = inputs.to(device)      # (B, 2, H, W)
        targets = targets.to(device)    # (B, 1, H, W)

        # Make sure target has proper dim!
        targets = targets.unsqueeze(1) if targets.ndim == 3 else targets

        optimizer.zero_grad()
        outputs = model(inputs)         # (B, 1, H, W)

        if isinstance(outputs, list):
            outputs = outputs[0]

        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        loop.set_postfix({'loss': train_loss / ((loop.n + 1) * inputs.size(0))})
    
    train_loss /= len(train_loader.dataset)
    train_loss_on_epoch.append(train_loss)
    print(f'Epoch [{epoch + 1}/{num_epochs}] - Train Loss: {train_loss:.6f}')

    # Validation loop
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            outputs = model(inputs)

            if isinstance(outputs, list):
                outputs = outputs[0]

            loss = criterion(outputs, targets)
            val_loss += loss.item() * inputs.size(0)

        val_loss /= len(val_loader.dataset)
        val_loss_on_epoch.append(val_loss)
        print(f'Epoch [{epoch + 1}/{num_epochs}] - Val Loss: {val_loss:.6f}')


In [ ]:
plot_curve(np.array(train_loss_on_epoch), np.arange(1, len(train_loss_on_epoch) + 1, 1), 'Training Loss')
plot_curve(np.array(val_loss_on_epoch), np.arange(1, len(val_loss_on_epoch) + 1, 1), 'Validation Loss')

In [ ]:
# Some random test
input_example, output_example = train_dataset[4]

output_example = output_example.cpu().detach().numpy()

In [ ]:
plot_2d_array(output_example, 'Ground Truth', cmap='viridis', min=np.min(output_example), max=np.max(output_example))

In [ ]:
model.eval()  # set to evaluation mode
with torch.no_grad():
    input_example_gpu = input_example.to(device)
    input_example_batched = input_example_gpu.unsqueeze(0)

    velocity_pred = model(input_example_batched).cpu().detach().numpy().squeeze().squeeze()

    plot_2d_array(velocity_pred, 'Prediction', cmap='viridis', min=np.min(velocity_pred), max=np.max(velocity_pred))